# TRCG-Fusion: architecture and evaluation

This notebook is a companion to the completed Stages 1–12. It imports the
actual model and evaluation code and reads the saved aggregate experiment
results. No training runs or data extraction are started.

The final calibrated five-seed ensemble achieved AUROC 0.8134 and AUPRC
0.4464, compared with 0.8131 and 0.4459 for EHR-only. Brier score and log
loss improved, while paired intervals for AUROC and AUPRC included zero.
The test evaluation is retrospective: this benchmark had already been
inspected during earlier project stages.

## Setup

Install `requirements.txt` and `requirements-notebook.txt`, and select that
Python environment as the kernel. The default walkthrough needs only the
repository code and aggregate files in `results/`. It does not load patient
records, images, feature caches, or trained checkpoints. The final section
is an optional local inference workflow, disabled by default.

In [1]:
from pathlib import Path
import importlib.util
import inspect
import json
import sys
import numpy as np
import pandas as pd
import torch
from IPython.display import Code, Markdown, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/models/trcg_encoders.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository or its notebooks folder.')
for folder in (ROOT, ROOT / 'scripts'):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

def load_script(filename, module_name):
    spec = importlib.util.spec_from_file_location(module_name, ROOT / 'scripts' / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

def table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |',
             '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines += ['| ' + ' | '.join(map(str, row)) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))

def show_source(obj):
    display(Code(inspect.getsource(obj), language='python'))

from src.models.trcg_encoders import TimeAwareEHR, EHRMortality, CXRMortality
from src.models.trcg_fusion import AlignmentProjectors, TRCGFusion
stage7 = load_script('18_train_stage7_refinement.py', 'review_stage7')
stage3 = load_script('08_train_temporal_alignment.py', 'review_stage3')
final = load_script('36_stage12_finalize.py', 'review_final')
print('Imported the research implementation. No models trained or checkpoints loaded.')

Imported the research implementation. No models trained or checkpoints loaded.


## Encoders and temporal alignment

`TimeAwareEHR` combines values, observation masks, time since observation,
categorical embeddings, and continuous time embeddings. Causal attention
prevents an EHR state at image time from seeing later observations. The
encoder uses 17 variables, a 128-dimensional state, two layers, and four
attention heads. `CXRMortality` uses a ResNet34 with a 512-dimensional state.
ImageNet weights are loaded from a local file during the experiment.

The following cells show code directly from the repository, rather than a
second implementation. No randomly initialized model is used as an example
of trained performance.

In [2]:
table(['Component', 'Implementation', 'State dimension'], [
    ['EHR', 'src/models/trcg_encoders.py: TimeAwareEHR', '128'],
    ['CXR', 'src/models/trcg_encoders.py: CXRMortality', '512'],
    ['Alignment', 'src/models/trcg_fusion.py: AlignmentProjectors', '128 per modality'],
    ['Final fusion', 'scripts/18_train_stage7_refinement.py: SafeTRCG', '128'],
])
show_source(TimeAwareEHR.forward)
show_source(CXRMortality)

| Component | Implementation | State dimension |
| --- | --- | --- |
| EHR | src/models/trcg_encoders.py: TimeAwareEHR | 128 |
| CXR | src/models/trcg_encoders.py: CXRMortality | 512 |
| Alignment | src/models/trcg_fusion.py: AlignmentProjectors | 128 per modality |
| Final fusion | scripts/18_train_stage7_refinement.py: SafeTRCG | 128 |

def forward(self, batch, return_sequence=False):
        x = torch.einsum(
            "btv,vd->btd",
            batch["values"],
            self.value_basis,
        )

        x = x + torch.einsum(
            "btv,vd->btd",
            batch["mask"],
            self.mask_basis,
        )

        x = x + torch.einsum(
            "btv,vd->btd",
            batch["delta"],
            self.delta_basis,
        )

        for name, idx in self.cat_indices.items():
            x = x + self.cat_embeddings[name](
                batch["categories"][..., idx]
            )

        x = self.in_norm(
            x + self.time_embedding(batch["times"])
        )

        seq_len = x.shape[1]

        # Critical: an earlier EHR state must not see
        # observations that happened later.
        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=x.device,
                dtype=torch.bool,
            ),
            diagonal=1,
        )

        lengths = batch["lengths"]

        padding_mask = (
            torch.arange(
                seq_len,
                device=x.device,
            )[None, :]
            >= lengths[:, None]
        )

        h = self.out_norm(
            self.encoder(
                x,
                mask=causal_mask,
                src_key_padding_mask=padding_mask,
            )
        )

        final = h[
            torch.arange(
                len(lengths),
                device=x.device,
            ),
            lengths - 1,
        ]

        if return_sequence:
            return final, h

        return final

class CXRMortality(nn.Module):
    def __init__(self, imagenet_checkpoint):
        super().__init__()

        net = models.resnet34(weights=None)

        if imagenet_checkpoint is not None:
            weights = torch.load(
                imagenet_checkpoint,
                map_location="cpu",
                weights_only=True,
            )
            net.load_state_dict(weights)

        net.fc = nn.Identity()

        self.encoder = net
        self.head = nn.Linear(512, 1)

    def forward(self, image):
        features = self.encoder(image)
        return self.head(features).squeeze(-1)

### Shared projection space

Project the causal EHR anchor and the CXR state into a common space and
normalize them. The Stage 3 loss uses paired examples in both directions.
Stages 11–12 fit separate projectors for independently trained encoder
seeds. Feature extraction and contrastive training require local MIMIC
inputs and checkpoints; only the implementation is inspected here.

In [3]:
show_source(AlignmentProjectors)
show_source(stage3.contrastive_loss)

class AlignmentProjectors(nn.Module):
    """Matches Stage-3 AlignmentProjectors state_dict keys exactly."""
    def __init__(self):
        super().__init__()
        self.ehr = nn.Sequential(nn.LayerNorm(128), nn.Linear(128, 128),
                                 nn.GELU(), nn.Linear(128, 128))
        self.cxr = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, 128),
                                 nn.GELU(), nn.Linear(128, 128))
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def forward(self, ehr, cxr):
        return (F.normalize(self.ehr(ehr), dim=-1),
                F.normalize(self.cxr(cxr), dim=-1))

def contrastive_loss(e, c, logit_scale):
    scale = logit_scale.exp().clamp(max=100)
    similarity = scale * (e @ c.T)
    target = torch.arange(len(e), device=e.device)
    return 0.5 * (
        F.cross_entropy(similarity, target)
        + F.cross_entropy(similarity.T, target)
    ), similarity

## Reliability and gated fusion

The original fusion uses the 48-hour EHR state, a pre-CXR EHR anchor, the
image state, image age, anchor validity, five EHR quality descriptors, and
image-prediction entropy. EHR quality captures observation coverage and
timing. Learned EHR and image reliability scores enter the gate; the image
reliability score also scales the residual directly.

In the final `safe_age` variant, the learned clinical-change branch is
replaced with zeros. The mortality heads are frozen, the residual output
layer starts at zero, and an invalid anchor makes the correction zero.
The correction is `gamma * r_cxr * anchor_valid * delta`. The age penalty
uses the observed difference between EHR states, not the disabled learned
change branch. Reliability scores are learned internal weights, not
independently validated clinical quality probabilities.

In [4]:
show_source(stage7.SafeTRCG.forward)
show_source(stage7.objective)

def forward(self, b):
        zt, za, zc = b['ehr_at_48'], b['ehr_at_cxr'], b['cxr']
        age = (b['cxr_age'].view(-1, 1) / 48.).clamp(0, 1)
        valid = b['anchor_valid'].view(-1, 1).float()
        entropy = b['cxr_entropy'].view(-1, 1)
        qe, qc = self.projections(za, zc)
        agreement = torch.cat([qc, qe, (qc-qe).abs(), qc*qe], dim=-1)
        d = torch.zeros_like(zt)  # remove learned post-CXR change component
        re = torch.sigmoid(self.ehr_reliability(b['ehr_quality']))
        rc = torch.sigmoid(self.cxr_reliability(torch.cat(
            [age, entropy, self.image_quality(zc)], dim=-1)))
        delta = self.residual(torch.cat([zc, za, d, agreement], dim=-1))
        if self.variant == 'safe_fixed':
            gamma = torch.full_like(age, self.constant_gamma)
        else:
            gamma = torch.sigmoid(self.gate(torch.cat(
                [agreement, d, age, re, rc, valid], dim=-1)))
        # Missing pre-CXR EHR anchor: avoid invalid cross-modal concordance.
        effective = gamma * rc * valid
        fused = zt + effective * delta
        fused_logit = self.ehr_head(fused).squeeze(-1)
        ehr_logit = self.ehr_head(zt).squeeze(-1)
        return dict(logits=fused_logit, ehr_logits=ehr_logit,
                    gamma=gamma.squeeze(-1), r_cxr=rc.squeeze(-1),
                    effective=effective.squeeze(-1), delta=delta)

def objective(model_output, x, y, pos_weight, variant):
    mort = nn.functional.binary_cross_entropy_with_logits(
        model_output['logits'], y, pos_weight=pos_weight)
    penalty = mort.new_zeros(())
    if variant == 'safe_age':
        with torch.no_grad():
            clinical_change = (x['ehr_at_48'] - x['ehr_at_cxr']).norm(dim=-1)
            clinical_change = clinical_change / (clinical_change + 3.)
            age = (x['cxr_age'] / 48.).clamp(0, 1)
            prior = age * clinical_change * x['anchor_valid']
        penalty = (model_output['gamma'] * prior).mean()
    return mort + (0.02 * penalty if variant == 'safe_age' else 0.), mort, penalty

## Evaluation protocol

The original split has 4,956 training, 490 validation, and 1,231 test stays.
Stage 12 uses seeds 42–46. Each seed has its own EHR/CXR encoders, alignment
projectors, and matched `safe_age` fusion head. Seeds share the same patient
split and ImageNet initialization.

First average the five predicted probabilities. Then apply one temperature
per ensemble, fitted by unweighted validation negative log likelihood.
Temperatures remain fixed during test evaluation. The following functions
are the actual implementation used for those steps; they are not invoked
on patient-level data in this walkthrough.

In [5]:
show_source(final.metrics)
show_source(final.fit_temperature)
show_source(final.apply_temperature)

report = json.loads((ROOT / 'results/trcg_stage12_final_test_summary.json').read_text())
locked = json.loads((ROOT / 'results/trcg_stage12_locked_temperatures.json').read_text())
validation = json.loads((ROOT / 'results/trcg_stage12_validation_summary.json').read_text())
assert report['seeds'] == locked['seeds'] == [42, 43, 44, 45, 46]
assert locked['source'] == 'validation_only'
assert report['scope'] == 'retrospective_benchmark_test'
assert report['n'] == 1231
for model in ('ehr', 'fusion'):
    assert report['ensemble'][model]['temperature'] == locked['temperatures'][model]
    assert validation['validation'][model]['temperature'] == locked['temperatures'][model]
table(['Ensemble', 'Locked validation temperature'], [
    [name, f"{locked['temperatures'][name]:.6f}"] for name in ('ehr', 'fusion')
])

def metrics(y,p):
    return dict(auprc=float(average_precision_score(y,p)),auroc=float(roc_auc_score(y,p)),
        brier=float(brier_score_loss(y,p)),log_loss=float(log_loss(y,p,labels=[0,1])))

def fit_temperature(y,p):
    if set(np.unique(y))!={0,1}:raise RuntimeError('Both classes needed for temperature fitting')
    logits=logit(np.clip(p,1e-6,1-1e-6))
    def loss(t):
        pred=expit(logits/t)
        return log_loss(y,np.clip(pred,1e-7,1-1e-7),labels=[0,1])
    result=minimize_scalar(loss,bounds=(0.5,10.),method='bounded',options={'xatol':1e-5})
    if not result.success:raise RuntimeError('Temperature fitting failed')
    return float(result.x)

def apply_temperature(p,t):
    return expit(logit(np.clip(p,1e-6,1-1e-6))/t)

| Ensemble | Locked validation temperature |
| --- | --- |
| ehr | 1.021991 |
| fusion | 1.043674 |

## Saved Stage 12 test results

Read the full-precision aggregate JSON from the completed run. These are
calibrated ensemble metrics. They are not computed by averaging the
individual seed AUROCs or AUPRCs. Lower Brier score and log loss are better.

In [6]:
metric_names = [('auroc', 'AUROC'), ('auprc', 'AUPRC'),
                ('brier', 'Brier score'), ('log_loss', 'Log loss')]
table(['Metric', 'EHR-only', 'TRCG-Fusion', 'Fusion minus EHR'], [
    [label,
     f"{report['ensemble']['ehr']['calibrated'][key]:.4f}",
     f"{report['ensemble']['fusion']['calibrated'][key]:.4f}",
     f"{report['ensemble']['fusion']['calibrated'][key] - report['ensemble']['ehr']['calibrated'][key]:+.6f}"]
    for key, label in metric_names
])

| Metric | EHR-only | TRCG-Fusion | Fusion minus EHR |
| --- | --- | --- | --- |
| AUROC | 0.8131 | 0.8134 | +0.000312 |
| AUPRC | 0.4459 | 0.4464 | +0.000503 |
| Brier score | 0.1109 | 0.1085 | -0.002418 |
| Log loss | 0.3608 | 0.3536 | -0.007139 |

### Paired uncertainty

The saved intervals use 1,500 subject-cluster bootstrap resamples of the
same test cohort, comparing the calibrated ensemble predictions. The
implementation resamples subjects and keeps all stays for each sampled
subject together. These intervals do not account for model selection or
dataset shift. A table is used here because the metrics have different
scales and the AUROC/AUPRC differences are small.

In [7]:
bootstrap = report['paired_cluster_bootstrap']
assert bootstrap['method'] == 'subject_cluster_bootstrap'
assert bootstrap['replicates'] == 1500
rows = []
for metric in ('auprc', 'auroc', 'brier'):
    lo, hi = bootstrap[f'fusion_minus_ehr_{metric}_95ci']
    rows.append([metric.upper(), f'[{lo:+.6f}, {hi:+.6f}]', lo <= 0 <= hi])
table(['Difference', '95% interval: fusion minus EHR', 'Includes zero'], rows)
show_source(final.cluster_bootstrap)

| Difference | 95% interval: fusion minus EHR | Includes zero |
| --- | --- | --- |
| AUPRC | [-0.002454, +0.003837] | True |
| AUROC | [-0.000514, +0.001059] | True |
| BRIER | [-0.003190, -0.001634] | False |

def cluster_bootstrap(df,predA,predB,replicates=1500):
    ids=df['subject_id'].astype(str).to_numpy()
    y=df['label'].to_numpy(int)
    subs=np.unique(ids)
    locations={s:np.flatnonzero(ids==s) for s in subs}
    rng=np.random.default_rng(12046)
    diffs=[];da=[];db=[]
    a=df[predA].to_numpy(float);b=df[predB].to_numpy(float)
    for _ in range(replicates):
        chosen=rng.choice(subs,size=len(subs),replace=True)
        index=np.concatenate([locations[s] for s in chosen])
        labels=y[index]
        if len(np.unique(labels))<2:continue
        diffs.append(average_precision_score(labels,a[index])-average_precision_score(labels,b[index]))
        da.append(roc_auc_score(labels,a[index])-roc_auc_score(labels,b[index]))
        db.append(brier_score_loss(labels,a[index])-brier_score_loss(labels,b[index]))
    if len(diffs)<int(replicates*.95):raise RuntimeError('Insufficient valid cluster resamples')
    return dict(method='subject_cluster_bootstrap',replicates=len(diffs),seed=12046,
        fusion_minus_ehr_auprc_95ci=np.quantile(diffs,[.025,.975]).tolist(),
        fusion_minus_ehr_auroc_95ci=np.quantile(da,[.025,.975]).tolist(),
        fusion_minus_ehr_brier_95ci=np.quantile(db,[.025,.975]).tolist())

Fusion slightly improved probabilistic accuracy. The AUROC and AUPRC
intervals include zero, so these results do not demonstrate a statistically
reliable improvement in discrimination. The Brier interval favors fusion.
This supports a narrow conclusion about these retrospective predictions,
not a claim of clinical benefit.

### Individual seeds

The following results are uncalibrated individual models, copied from the
existing aggregate per-seed CSV. They measure run variation on one fixed
split and should not be compared directly with calibrated ensemble scores
as if they used the same aggregation procedure.

In [8]:
per_seed = pd.read_csv(ROOT / 'results/trcg_stage12_per_seed_test.csv')
assert set(per_seed.columns) == {'seed', 'model', 'auprc', 'auroc', 'brier', 'log_loss'}
assert len(per_seed) == 10
reference = pd.DataFrame(report['per_seed']).sort_values(['seed', 'model']).reset_index(drop=True)
pd.testing.assert_frame_equal(
    per_seed.sort_values(['seed', 'model']).reset_index(drop=True)[reference.columns],
    reference, check_exact=False, rtol=1e-12, atol=1e-12,
)
table(['Seed', 'Model', 'AUROC', 'AUPRC', 'Brier', 'Log loss'], [
    [int(row.seed), row.model, f'{row.auroc:.4f}', f'{row.auprc:.4f}',
     f'{row.brier:.4f}', f'{row.log_loss:.4f}']
    for row in per_seed.itertuples(index=False)
])

| Seed | Model | AUROC | AUPRC | Brier | Log loss |
| --- | --- | --- | --- | --- | --- |
| 42 | ehr | 0.8065 | 0.4341 | 0.1337 | 0.4164 |
| 42 | fusion | 0.8065 | 0.4336 | 0.1321 | 0.4118 |
| 43 | ehr | 0.7900 | 0.4080 | 0.1138 | 0.3762 |
| 43 | fusion | 0.7903 | 0.4085 | 0.1130 | 0.3745 |
| 44 | ehr | 0.7732 | 0.3784 | 0.1561 | 0.5823 |
| 44 | fusion | 0.7731 | 0.3790 | 0.1547 | 0.5750 |
| 45 | ehr | 0.7977 | 0.4239 | 0.1140 | 0.3779 |
| 45 | fusion | 0.7989 | 0.4252 | 0.1111 | 0.3770 |
| 46 | ehr | 0.8113 | 0.4660 | 0.1123 | 0.3681 |
| 46 | fusion | 0.8113 | 0.4660 | 0.1119 | 0.3671 |

## Earlier experiments

Stages 1–5 establish the cohort, unimodal encoders, temporal alignment,
frozen and fine-tuned fusion, and calibration. Stages 6–8 compare component
ablations, safe gates, and a DenseNet alternative. Stages 9–10 inspect image
contribution and shuffle controls. Their validation comparisons are
exploratory because validation was used during model development. In
Stages 6–10, different fusion seeds share encoders; they are not the
independent encoder runs used in Stages 11–12.

In [9]:
for filename in ('trcg_stage6b_validation_comparison.csv',
                 'trcg_stage7_validation_comparison.csv',
                 'trcg_stage8_validation_comparison.csv'):
    comparison = pd.read_csv(ROOT / 'results' / filename)
    display(Markdown('### ' + filename))
    table(list(comparison.columns), [
        [f'{value:.6f}' if isinstance(value, float) else value for value in row]
        for row in comparison.itertuples(index=False, name=None)
    ])

### trcg_stage6b_validation_comparison.csv

| variant | runs | val_auprc_mean | val_auprc_std | val_auroc_mean |
| --- | --- | --- | --- | --- |
| ehr_only | 3 | 0.551131 | 0.001102 | 0.844650 |
| no_change | 3 | 0.541598 | 0.001995 | 0.854308 |
| no_align_loss | 3 | 0.527757 | 0.003206 | 0.860495 |
| full | 3 | 0.527509 | 0.003298 | 0.859937 |
| no_stage3_init | 3 | 0.527236 | 0.003568 | 0.859893 |
| no_reliability | 3 | 0.522290 | 0.001689 | 0.861371 |
| no_gate | 3 | 0.507582 | 0.008597 | 0.855316 |

### trcg_stage7_validation_comparison.csv

| variant | seeds | val_auprc_mean | val_auprc_sd | val_auroc_mean | ehr_base_auprc_mean |
| --- | --- | --- | --- | --- | --- |
| safe_learned | 3 | 0.549309 | 0.000693 | 0.847289 | 0.546326 |
| safe_fixed | 3 | 0.545626 | 0.000917 | 0.849118 | 0.546326 |
| safe_age | 3 | 0.549804 | 0.000091 | 0.847147 | 0.546326 |

### trcg_stage8_validation_comparison.csv

| variant | n | mean_val_auprc | std_val_auprc | mean_val_auroc | std_val_auroc |
| --- | --- | --- | --- | --- | --- |
| dense_learned | 3 | 0.546649 | 0.000335 | 0.845833 | 0.000164 |
| dense_fixed | 3 | 0.548059 | 0.000511 | 0.845931 | 0.000033 |
| dense_age | 3 | 0.546549 | 0.000129 | 0.845647 | 0.000050 |

## Optional local inference — MIMIC caches and checkpoints required

This section is disabled by default. To use it, open this notebook inside
your existing local research project, with the original `cache/` and
`checkpoints/` folders present. It reads the five seed-specific Stage 12
test feature files and the EHR, CXR, alignment, and fusion checkpoints. It
does not train models, extract data, refit temperatures, bootstrap, or
overwrite the saved experiment outputs. It displays only aggregate metrics.

`predict()` internally handles private IDs, labels, and probabilities.
Keep those in memory; do not display or export patient-level tables in a
notebook intended for sharing. The published notebook was executed with
`RUN_LOCAL_INFERENCE = False`.

In [10]:
RUN_LOCAL_INFERENCE = False

if RUN_LOCAL_INFERENCE:
    predictions = final.prediction_frame('test')
    rows = []
    for name in ('ehr', 'fusion'):
        probability = final.apply_temperature(
            predictions[f'{name}_mean'].to_numpy(), locked['temperatures'][name]
        )
        observed = final.metrics(predictions['label'].to_numpy(), probability)
        expected = report['ensemble'][name]['calibrated']
        for metric, _ in metric_names:
            np.testing.assert_allclose(observed[metric], expected[metric], rtol=1e-5, atol=1e-6)
        rows.append([name] + [f'{observed[key]:.6f}' for key, _ in metric_names])
    table(['Model', 'AUROC', 'AUPRC', 'Brier score', 'Log loss'], rows)
    del predictions, probability
else:
    print('Skipped: local patient-level feature caches and trained checkpoints are required.')

Skipped: local patient-level feature caches and trained checkpoints are required.


## Checks and limits

The default cells verify that the stored temperatures match the validation
artifact, that the seed list and test scope are correct, and that the
per-seed CSV matches the final JSON. They inspect actual model/evaluation
source and display actual saved metrics. They do not independently
recompute metrics from private patient predictions. Reproduction of those
metrics requires the optional local inference section.

Repeated validation inspection and earlier test inspection limit the
strength of the result. All seeds share a cohort and ImageNet starting
weights, and there is no external hospital validation. See the README and
the caveats in the final aggregate JSON for the scope of the conclusions.